# IRE Assignment 1 + 2 — Full Pipeline
## Kaggle T4 GPU Notebook

**Run cells in order. Do NOT skip cells.**

### Datasets (already in the working dir from A1 run):
- MIND: downloaded from HuggingFace (cells 7-10 if needed)
- EB-NeRD: downloaded from S3 (cell 11 if needed)

### What to download after each step:
| After Cell | Download file | Upload to |
|---|---|---|
| Cell 14 | `data/results/bm25_mind_val.json` | Paste to chat |
| Cell 15 | `data/results/semantic_mind_val.json` | Paste to chat |
| Cell 16 | `data/results/bm25_ebnerd_val.json` | Paste to chat |
| Cell 17 | `data/results/semantic_ebnerd_val.json` | Paste to chat |
| Cell 18 | LightGBM training output (printed) | Paste to chat |
| Cell 19 | LightGBM training output (printed) | Paste to chat |
| Cell 20 | `data/results/eval_mind_val_lgbm.json` | Paste to chat |
| Cell 21 | `data/results/eval_ebnerd_val_lgbm.json` | Paste to chat |
| Cell 22 | NRMS training output (printed) | Paste to chat |
| Cell 23 | Ablation output (printed) | Paste to chat |
| Cell 24 | `data/results/serving_mind.json` | Paste to chat |
| Cell 25 | `mind_submission.zip`, `ebnerd_submission.zip` | Upload to Codabench |
| Cell 26 | Print ALL results JSON files | Paste everything to chat |

In [ ]:
# CELL 1: Check disk and environment
import os
print('=== Disk space ===')
os.system('df -h /kaggle/working')

print('\n=== GPU ===')
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\n=== Input files ===')
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# CELL 2: Clone / update the repo
import os

if not os.path.exists('/kaggle/working/news-retrieval-system'):
    !git clone https://github.com/imchaitanya0/news-retrieval-system.git
else:
    !cd /kaggle/working/news-retrieval-system && git pull origin main
    print('Repo already exists — pulled latest')

In [ ]:
# CELL 3: Set working directory + Python path
import os, sys

%cd /kaggle/working/news-retrieval-system
sys.path.insert(0, '/kaggle/working/news-retrieval-system')

# Create required directories
for d in [
    'data/raw/mind', 'data/raw/ebnerd',
    'data/processed',
    'data/feature_store/embeddings/mind',
    'data/feature_store/embeddings/ebnerd',
    'data/feature_store/bm25/mind',
    'data/feature_store/bm25/ebnerd',
    'data/feature_store/semantic/mind',
    'data/feature_store/semantic/ebnerd',
    'data/models', 'data/results', 'data/submissions',
]:
    os.makedirs(d, exist_ok=True)

print('Working dir:', os.getcwd())
print('✓ Directories created')

In [ ]:
# CELL 4: Install dependencies
# faiss-gpu for T4, bm25s for fast BM25, sentence-transformers for embeddings
!pip install -q huggingface_hub
!pip install -q faiss-gpu bm25s rank_bm25 sentence-transformers lightgbm polars tqdm scikit-learn

In [ ]:
# CELL 5: HuggingFace login (needed to download MIND dataset)
# !! REPLACE the token below with your actual HF token from https://huggingface.co/settings/tokens
!pip install -q huggingface_hub
from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')  # <-- your token from A1

In [ ]:
# CELL 6: Download MIND Large datasets from HuggingFace
# SKIP this cell if data/raw/mind/ already has train/ val/ with behaviors.tsv
import os
from pathlib import Path

# Check if already downloaded
mind_train_exists = Path('data/raw/mind/train/behaviors.tsv').exists() or \
                    Path('data/raw/mind/MINDlarge_train/behaviors.tsv').exists()
print(f'MIND train already downloaded: {mind_train_exists}')

if not mind_train_exists:
    from huggingface_hub import hf_hub_download
    
    Path('data/raw/mind/train').mkdir(parents=True, exist_ok=True)
    Path('data/raw/mind/val').mkdir(parents=True, exist_ok=True)

    print('Downloading MINDlarge_train.zip ...')
    train_path = hf_hub_download(
        repo_id='yjw1029/MIND',
        filename='MINDlarge_train.zip',
        repo_type='dataset',
        local_dir='data/raw/mind/train',
    )

    print('Downloading MINDlarge_dev.zip ...')
    dev_path = hf_hub_download(
        repo_id='yjw1029/MIND',
        filename='MINDlarge_dev.zip',
        repo_type='dataset',
        local_dir='data/raw/mind/val',
    )

    print('Downloading MINDlarge_test.zip ...')
    test_path = hf_hub_download(
        repo_id='yjw1029/MIND',
        filename='MINDlarge_test.zip',
        repo_type='dataset',
        local_dir='data/raw/mind',
    )
    print('✓ Downloads complete')
else:
    print('✓ MIND already downloaded, skipping')

In [ ]:
# CELL 7: Extract MIND zips
# SKIP this cell if you already extracted them in A1 run
import zipfile, os
from pathlib import Path
from tqdm import tqdm

zip_jobs = [
    ('data/raw/mind/train/MINDlarge_train.zip', 'data/raw/mind/train'),
    ('data/raw/mind/val/MINDlarge_dev.zip',     'data/raw/mind/val'),
    ('data/raw/mind/MINDlarge_test.zip',        'data/raw/mind/test'),
]

for zip_path, extract_dir in zip_jobs:
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)
    # Check if already extracted
    if (extract_dir / 'behaviors.tsv').exists():
        print(f'✓ Already extracted: {extract_dir}')
        continue
    if not zip_path.exists():
        print(f'✗ Zip not found: {zip_path} — skip')
        continue
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {zip_path.name} → {extract_dir} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in tqdm(z.namelist(), desc=zip_path.stem):
            z.extract(member, extract_dir)
    print(f'  ✓ Done')

In [ ]:
# CELL 8: Download EB-NeRD (small version) from S3
# SKIP if already downloaded from A1 run
import requests, os, zipfile
from tqdm import tqdm
from pathlib import Path

urls = {
    'ebnerd_small.zip':   'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_small.zip',
    'ebnerd_testset.zip': 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip',
}

# Check if already extracted
already_done = Path('data/raw/ebnerd/train/behaviors.parquet').exists()
print(f'EB-NeRD already extracted: {already_done}')

if not already_done:
    for fname, url in urls.items():
        dest = fname
        if os.path.exists(dest):
            print(f'{fname} already downloaded, skipping.')
        else:
            print(f'Downloading {fname} ...')
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0))
                with open(dest, 'wb') as out_f:
                    with tqdm(total=total, unit='B', unit_scale=True, desc=fname) as bar:
                        for chunk in r.iter_content(chunk_size=8192):
                            out_f.write(chunk)
                            bar.update(len(chunk))

    # Extract
    for fname, extract_to in [('ebnerd_small.zip', 'data/raw/ebnerd'), ('ebnerd_testset.zip', 'data/raw/ebnerd')]:
        if os.path.exists(fname):
            print(f'Extracting {fname} ...')
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall(extract_to)
            print(f'  ✓ Extracted to {extract_to}')
else:
    print('✓ EB-NeRD already extracted, skipping')

In [ ]:
# CELL 9: Verify raw data structure
import os
from pathlib import Path

print('=== MIND raw structure ===')
for root, dirs, files in os.walk('data/raw/mind'):
    depth = root.replace('data/raw/mind', '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth <= 2:
        for f in sorted(files)[:5]:
            size_mb = Path(os.path.join(root, f)).stat().st_size / 1e6
            print(f'{indent}  {f}  ({size_mb:.1f} MB)')

print('\n=== EB-NeRD raw structure ===')
for root, dirs, files in os.walk('data/raw/ebnerd'):
    depth = root.replace('data/raw/ebnerd', '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files)[:5]:
        size_mb = Path(os.path.join(root, f)).stat().st_size / 1e6
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')

In [ ]:
# CELL 10: Build processed parquets from raw data
# Output: data/processed/articles_mind.parquet, behaviors_mind_train/val/test.parquet
#         data/processed/articles_ebnerd.parquet, behaviors_ebnerd_train/val/test.parquet
!python -m src.data.build_pipeline

# Verify
import polars as pl
from pathlib import Path
print('\n=== Processed files ===')
for f in sorted(Path('data/processed').glob('*.parquet')):
    df = pl.read_parquet(f)
    print(f'  {f.name:50s}  {f.stat().st_size/1e6:6.1f} MB  {len(df):>8,} rows  cols={df.columns}')

In [ ]:
# CELL 11: Compute sentence-transformer embeddings (MIND)
# ⏱ ~20-30 min on T4. Cached after first run.
# Output: data/feature_store/embeddings/mind/article_embeddings.npy
import polars as pl
from src.retrieval.semantic import load_or_compute_embeddings

articles_mind = pl.read_parquet('data/processed/articles_mind.parquet')
print(f'MIND articles: {len(articles_mind):,}')
embs_mind, ids_mind = load_or_compute_embeddings(articles_mind, 'mind')
print(f'✓ MIND embeddings: {embs_mind.shape}  dtype={embs_mind.dtype}')

In [ ]:
# CELL 12: Compute sentence-transformer embeddings (EB-NeRD)
# ⏱ ~20-30 min on T4. Cached after first run.
# Output: data/feature_store/embeddings/ebnerd/article_embeddings.npy
import polars as pl
from src.retrieval.semantic import load_or_compute_embeddings

articles_ebnerd = pl.read_parquet('data/processed/articles_ebnerd.parquet')
print(f'EB-NeRD articles: {len(articles_ebnerd):,}')
embs_ebnerd, ids_ebnerd = load_or_compute_embeddings(articles_ebnerd, 'ebnerd')
print(f'✓ EB-NeRD embeddings: {embs_ebnerd.shape}  dtype={embs_ebnerd.dtype}')

In [ ]:
# CELL 13: Pull latest code (in case you re-run later)
!git pull origin main

In [ ]:
# CELL 14: BM25 Recall@K — MIND
# ⏱ ~10-15 min
# 📥 DOWNLOAD AFTER: data/results/bm25_mind_val.json  → paste contents to chat
!python -m src.retrieval.bm25 --dataset mind --split val --k 50 100 200

In [ ]:
# CELL 15: Semantic Recall@K — MIND
# ⏱ ~5 min (embeddings already cached)
# 📥 DOWNLOAD AFTER: data/results/semantic_mind_val.json  → paste to chat
!python -m src.retrieval.semantic --dataset mind --split val --k 50 100 200

In [ ]:
# CELL 16: BM25 Recall@K — EB-NeRD
# ⏱ ~10-15 min
# 📥 DOWNLOAD AFTER: data/results/bm25_ebnerd_val.json  → paste to chat
!python -m src.retrieval.bm25 --dataset ebnerd --split val --k 50 100 200

In [ ]:
# CELL 17: Semantic Recall@K — EB-NeRD
# ⏱ ~5 min
# 📥 DOWNLOAD AFTER: data/results/semantic_ebnerd_val.json  → paste to chat
!python -m src.retrieval.semantic --dataset ebnerd --split val --k 50 100 200

In [ ]:
# CELL 18: Train LightGBM — MIND (8 features, LambdaMART)
# ⏱ ~30-60 min
# This auto-generates:
#   data/submissions/mind_val_lgbm.txt   (for evaluation)
#   data/submissions/mind_test_lgbm.zip  (for Codabench)
# 📥 COPY THE FULL OUTPUT (feature importances + val nDCG numbers) → paste to chat
!python -m src.ranking.train_lgbm --dataset mind --max-train-rows 500000

In [ ]:
# CELL 19: Train LightGBM — EB-NeRD
# ⏱ ~30-60 min
# 📥 COPY THE FULL OUTPUT → paste to chat
!python -m src.ranking.train_lgbm --dataset ebnerd --max-train-rows 500000

In [ ]:
# CELL 20: Full eval harness — MIND (AUC/MRR/nDCG/slices/CI)
# ⏱ ~5-10 min
# 📥 DOWNLOAD AFTER: data/results/eval_mind_val_lgbm.json  → paste to chat
!python -m src.evaluation.evaluate --dataset mind --strategy lgbm --split val

In [ ]:
# CELL 21: Full eval harness — EB-NeRD
# ⏱ ~5-10 min
# 📥 DOWNLOAD AFTER: data/results/eval_ebnerd_val_lgbm.json  → paste to chat
!python -m src.evaluation.evaluate --dataset ebnerd --strategy lgbm --split val

In [ ]:
# CELL 22: NRMS Baseline — MIND (A2 Q3)
# ⏱ ~45-60 min on T4 for 3 epochs
# Saves: data/models/nrms_mind.pt
#        data/results/nrms_mind_val.json
# 📥 COPY the epoch-by-epoch output → paste to chat
!python -m src.models.train_nrms --dataset mind --epochs 3 --max-train-rows 150000 --max-val-rows 10000

In [ ]:
# CELL 23: NRMS Baseline — EB-NeRD (A2 Q3)
# ⏱ ~45-60 min on T4 for 3 epochs
# 📥 COPY the epoch-by-epoch output → paste to chat
!python -m src.models.train_nrms --dataset ebnerd --epochs 3 --max-train-rows 150000 --max-val-rows 10000

In [ ]:
# CELL 24: Ablation Study — MIND + EB-NeRD (A2 Q3)
# ⏱ ~30-45 min (4 LightGBM variants × 2 datasets)
# 📥 COPY the full output → paste to chat
!python -m src.ranking.ablation --dataset mind --max-val-rows 20000
!python -m src.ranking.ablation --dataset ebnerd --max-val-rows 20000

In [ ]:
# CELL 25: Serving Benchmark (A2 Q4)
# ⏱ ~2-3 min
# 📥 COPY full output → paste to chat
!python -m src.evaluation.serving_benchmark --dataset mind --n-requests 500
!python -m src.evaluation.serving_benchmark --dataset ebnerd --n-requests 500

In [ ]:
# CELL 26: Package Codabench submission zips
# The zip files from train_lgbm are already created.
# This cell just verifies and re-packages if needed.
import os, zipfile, shutil
from pathlib import Path

os.chdir('/kaggle/working/news-retrieval-system')

# MIND — submission file must be named 'prediction.txt' (no 's') inside zip
mind_txt = Path('data/submissions/mind_test_lgbm.txt')
if mind_txt.exists():
    shutil.copy(mind_txt, '/kaggle/working/prediction.txt')
    with zipfile.ZipFile('/kaggle/working/mind_submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write('/kaggle/working/prediction.txt', arcname='prediction.txt')
    os.remove('/kaggle/working/prediction.txt')
    print('✓ mind_submission.zip created')
    os.system('unzip -l /kaggle/working/mind_submission.zip')
else:
    print('✗ mind_test_lgbm.txt not found. Run Cell 18 first.')

# EB-NeRD — submission file must be named 'predictions.txt' (WITH 's') inside zip
ebnerd_txt = Path('data/submissions/ebnerd_test_lgbm.txt')
if ebnerd_txt.exists():
    shutil.copy(ebnerd_txt, '/kaggle/working/predictions.txt')
    with zipfile.ZipFile('/kaggle/working/ebnerd_submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write('/kaggle/working/predictions.txt', arcname='predictions.txt')
    os.remove('/kaggle/working/predictions.txt')
    print('✓ ebnerd_submission.zip created')
    os.system('unzip -l /kaggle/working/ebnerd_submission.zip')
else:
    print('✗ ebnerd_test_lgbm.txt not found. Run Cell 19 first.')

In [ ]:
# CELL 27: PRINT ALL RESULTS — paste this entire output to chat
# I will use this to generate both design note PDFs + README
import json
from pathlib import Path

print('=' * 70)
print('COMPLETE RESULTS — PASTE THIS ENTIRE OUTPUT TO CHAT')
print('=' * 70)

result_files = sorted(Path('data/results').glob('*.json'))
if not result_files:
    print('No results files yet. Run earlier cells first.')
else:
    for f in result_files:
        print(f'\n{"=" * 60}')
        print(f'FILE: {f.name}')
        print('=' * 60)
        with open(f) as fp:
            print(json.dumps(json.load(fp), indent=2))

print('\n' + '=' * 70)
print('Submission zips in /kaggle/working/:')
for f in Path('/kaggle/working').glob('*.zip'):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')